### This notebook is for the plots of the enformer class distribution
- log(RNA/DNA) plot per enformer high vs random and enformer low vs random

In [1]:
import pandas as pd 
import numpy as np
import os
import yaml
import ast # convert string back to list: ast.literal_eval

import seaborn as sns
import matplotlib.pyplot as plt

# config 
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [16]:
# load bc_MPRAlm results
bc_MPRAlm_results = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm'], sep="\t")
# for this analysis we focus on tested variants
tested_bc_MPRAlm_results = bc_MPRAlm_results.loc[bc_MPRAlm_results['variant_id'].str.startswith('cardiac_neuro_cava_random')]
print('Number of variants after barcode MPRAlm: ', bc_MPRAlm_results.shape[0])
print('Number of tested variants after barcode MPRAlm: ', tested_bc_MPRAlm_results.shape[0])

# # load enformer count results (not needed right now)
# enformer_count_table = pd.read_csv(config['files']['creating']['enformer_mprasnakeflow_counts'], sep="\t")
# print('Number of prioritized variants from enformer which could be matched to counts tables of MPRAsnakeflow: ', enformer_count_table.shape[0])

# load raw enformer variant information
all_enformer_variants_table = pd.read_csv(config['files']['creating']['common_and_enformer_variants'], sep="\t", dtype={'DNase_max': float, 'max_col': 'string', 'enformer_class': 'string'})
all_enformer_variants_table['gene_type_list'] = all_enformer_variants_table['gene_type_list'].apply(ast.literal_eval)
# make string lists to panda lists
all_enformer_variants_table = all_enformer_variants_table.drop_duplicates(subset=['chr_pos_ref_alt', 'variant_type', 'enformer_class'])
print('Number of unique chr_pos_ref_alt variants in the enformer variant table: ', all_enformer_variants_table['chr_pos_ref_alt'].nunique())

Number of variants after barcode MPRAlm:  35039
Number of tested variants after barcode MPRAlm:  34525
Number of unique chr_pos_ref_alt variants in the enformer variant table:  68222


#### enformer_class numbers
- Without duplicate dropping:
    - enformer_high      24500
    - enformer_random     5250
    - enformer_low        5250
- with duplicate dropping: 
    - enformer_class
    - enformer_high      23811
    - enformer_random     5250
    - enformer_low        5041

In [17]:
all_enformer_variants_table['enformer_class'].value_counts()

enformer_class
enformer_high      23811
enformer_random     5250
enformer_low        5041
Name: count, dtype: Int64

#### enformer variant types: 
- ultra-rare    17053
- singleton     17049

In [131]:
all_enformer_variants_table['variant_type'].value_counts()

variant_type
ultra-rare    17053
singleton     17049
Name: count, dtype: int64

### Check if all chrom-pos-ref-alt ind enformer are unique
- No it is not 34100 unique `chr_pos_ref_alt` and 35000 is the shape
- I used the whole ID to merge multiple occurences of the same variant sometimes assigned to two genes (e.g. 14-23416213-C-T)
- Duplicates can be removed because the predictions used the full amount of sequence context
- found 2 cases where the enformer class is different for the same variant (3-41275463-A-G, 12-116214435-A-T)
- Since many similar variants what is the best strategy to merge them? 

In [78]:
all_enformer_variants_table_duplicates = all_enformer_variants_table_duplicates.loc[all_enformer_variants_table_duplicates['chr_pos_ref_alt'].duplicated(keep=False)]
all_enformer_variants_table_duplicates

,chrom,pos,id,ref,alt,DNase_max,max_col,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,variant_type,gene_type_list,enformer_class,chr_pos_ref_alt
9272,chr12,116214435,MED13L|ENSG00000123066.9|EH38E3043684|12-11621...,A,T,1.216711,502_DNASE:thyroid gland,chr12,116214435.0,MED13L|ENSG00000123066.9|EH38E3043684|12-11621...,A,T,1.0,PASS,AF=0.000191134;AC=29,ultra-rare,"[cardiac, neuro]",enformer_low,12-116214435-A-T
16636,chr3,41275463,CTNNB1|ENSG00000168036.18|EH38E3507688|3-41275...,A,G,180.593690,234_DNASE:HepG2,chr3,41275463.0,CTNNB1|ENSG00000168036.18|EH38E3507688|3-41275...,A,G,1.0,PASS,AF=1.31361e-05;AC=2,ultra-rare,"[cava, neuro]",enformer_high,3-41275463-A-G
28515,chr3,41275463,CTNNB1|ENSG00000168036.18|EH38E3507688|3-41275...,A,G,180.593690,234_DNASE:HepG2,chr3,41275463.0,CTNNB1|ENSG00000168036.18|EH38E3507688|3-41275...,A,G,1.0,PASS,AF=1.31361e-05;AC=2,ultra-rare,"[cava, neuro]",enformer_random,3-41275463-A-G
29243,chr12,116214435,MED13L|ENSG00000123066.9|EH38E3043684|12-11621...,A,T,1.216711,502_DNASE:thyroid gland,chr12,116214435.0,MED13L|ENSG00000123066.9|EH38E3043684|12-11621...,A,T,1.0,PASS,AF=0.000191134;AC=29,ultra-rare,"[cardiac, neuro]",enformer_random,12-116214435-A-T


In [8]:
all_enformer_variants_table['chr_pos_ref_alt'].nunique()

# get duplicates
chr_pos_ref_alt_value_counts = all_enformer_variants_table['chr_pos_ref_alt'].value_counts().reset_index()
not_unique_variants_enformer = chr_pos_ref_alt_value_counts.loc[chr_pos_ref_alt_value_counts['count'] > 1]
not_unique_variants_enformer['count'].value_counts()

count
2    818
3     41
Name: count, dtype: int64

In [60]:
def compare_count_and_gene_type_list(row):
    return len(row['gene_type_list']) == row['count']

In [65]:
print('Number of all duplicated chrom-pos-ref-alt: ', non_unique_count_gene_type_list_table.shape[0])
non_unique_count_gene_type_list_table = not_unique_variants_enformer.merge(all_enformer_variants_table[['chr_pos_ref_alt', 'gene_type_list']])

# check if len(gene_type_list) == count 
print('Number of duplicated chrom-pos-ref-alt because of multiple assignment: ', non_unique_count_gene_type_list_table.loc[non_unique_count_gene_type_list_table.apply(compare_count_and_gene_type_list, axis=1)].shape[0])

# investigate the other cases: 
non_unique_counts_without_assignment_problem = non_unique_count_gene_type_list_table.loc[~non_unique_count_gene_type_list_table.apply(compare_count_and_gene_type_list, axis=1)]
non_unique_counts_without_assignment_problem

Number of all duplicated chrom-pos-ref-alt:  1759
Number of duplicated chrom-pos-ref-alt because of multiple assignment:  1437


,chr_pos_ref_alt,count,gene_type_list
9,16-2032683-G-T,3,"[cardiac, neuro]"
10,16-2032683-G-T,3,[cava]
11,16-2032683-G-T,3,"[cardiac, neuro]"
24,16-2038426-C-T,3,"[cardiac, neuro]"
25,16-2038426-C-T,3,[cava]
...,...,...,...
1708,17-17840972-C-T,2,[neuro]
1709,22-42180570-T-C,2,[cava]
1710,22-42180570-T-C,2,[neuro]
1713,14-23416213-C-T,2,[cardiac]


### Make class distribution plot using barcode MPRAlm results

- check if all variant_ids have the variant pattern (all tested)
- merge using the chr_pos_ref_alt

In [18]:
import re
def has_variant_info_in_header(header):
    """Check if the header has the variant info (chr-pos-ref-alt) in the ending of the header"""
    pattern = r'([A-Z]|[0-9]+)-[0-9]+-[A-Z]-[A-Z]'
    matches = re.search(pattern, header)
    if matches:
        return True
    else: return False
    
    
def get_chrom_pos_ref_alt_pattern(header):
    pattern = r'([A-Z]|[0-9]+)-[0-9]+-[A-Z]-[A-Z]'
    matches = re.search(pattern, header)
    if matches:
        return matches.group()
    else: return "NA"

In [19]:
print('Number of barcode MPRAlm results: ', tested_bc_MPRAlm_results.shape[0])

Number of barcode MPRAlm results:  34525


In [20]:
print('Number of variant ids with chrom-pos-ref-alt pattern', tested_bc_MPRAlm_results['variant_id'].apply(has_variant_info_in_header).sum()) # 34525

Number of variant ids with chrom-pos-ref-alt pattern 34525


In [21]:
tested_bc_MPRAlm_results['chr_pos_ref_alt'] = tested_bc_MPRAlm_results['variant_id'].apply(get_chrom_pos_ref_alt_pattern)
print('Number of duplicated chr_pos_ref_alt in the tested variants ', tested_bc_MPRAlm_results['chr_pos_ref_alt'].duplicated().sum())
if tested_bc_MPRAlm_results['chr_pos_ref_alt'].duplicated().sum() != 0:
    Error

Number of duplicated chr_pos_ref_alt in the tested variants  0


/tmp/ipykernel_1294/3373732753.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_bc_MPRAlm_results['chr_pos_ref_alt'] = tested_bc_MPRAlm_results['variant_id'].apply(get_chrom_pos_ref_alt_pattern)


In [36]:
tested_bc_MPRAlm_results.loc[tested_bc_MPRAlm_results['chr_pos_ref_alt'] == "X-154029091-T-C"]
all_enformer_variants_table.loc[all_enformer_variants_table['chr_pos_ref_alt'] == "X-154029091-T-C"]

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,variant_type,gene_type_list,DNase_max,max_col,enformer_class,chr_pos_ref_alt
56143,chrX,154029091.0,MECP2|ENSG00000169057.25|EH38E3949372|X-154029...,T,C,1.0,InbreedingCoeff,AF=0.998663;AC=109831,common,[neuro],NaN,<NA>,<NA>,X-154029091-T-C


In [37]:
tested_bc_MPRAlm_enformer = tested_bc_MPRAlm_results.merge(all_enformer_variants_table, on='chr_pos_ref_alt', how='left')
print('Shape after merge: ', tested_bc_MPRAlm_enformer.shape[0])
tested_bc_MPRAlm_enformer_not_merged = tested_bc_MPRAlm_enformer.loc[tested_bc_MPRAlm_enformer['variant_type'].isna()]
print('Not mergable: ', tested_bc_MPRAlm_enformer_not_merged.shape[0]) # 0

tested_bc_MPRAlm_enformer_merged = tested_bc_MPRAlm_enformer.loc[~tested_bc_MPRAlm_enformer['variant_type'].isna()]
print('Mergable: ', tested_bc_MPRAlm_enformer_merged.shape[0])

Shape after merge:  34526
Not mergable:  0
Mergable:  34526


This means 18407 from the initial 34526 variants can be matched back to the initially prioritized variant set
- That is odd, I would expect, that all of these variants can be matched with the results from enformer
- Something stinks but I need to work with the data I have

## Do we have more singleton and ultra-rare variants among the significant results#
- for the significants_level 0.05 we only have singleton and ultra-rare variants
- yes: 
    - common        151
    - singleton     126
    - ultra-rare    113

In [38]:
tested_bc_MPRAlm_enformer_merged['variant_type'].value_counts()

variant_type
common        16119
singleton      9306
ultra-rare     9101
Name: count, dtype: int64

In [43]:
singificants_niveau = 0.05
matched_significant_results = tested_bc_MPRAlm_enformer_merged.loc[tested_bc_MPRAlm_enformer_merged['adj.P.Val'] < singificants_niveau]
print(matched_significant_results.shape[0]) # 390
matched_significant_results['variant_type'].value_counts()

390


variant_type
common        151
singleton     126
ultra-rare    113
Name: count, dtype: int64

It was shown already that rare variants show higher effects but investigate the enformer predictions (might have influenced the prediction)
I will focus only on the singleton and ultra-rare and look in enformers prediction performance: Is there a significant enrichment of enformer high in the significant group

In [59]:
rare_variant_results = tested_bc_MPRAlm_enformer_merged[~tested_bc_MPRAlm_enformer_merged['enformer_class'].isna()]
matched_significant_results_rare = matched_significant_results[~matched_significant_results['enformer_class'].isna()]
matched_significant_results_rare['variant_type'].value_counts()

variant_type
singleton     126
ultra-rare    113
Name: count, dtype: int64

In [60]:
rare_variant_results['enformer_class'].value_counts() # 18407 


enformer_class
enformer_high      13500
enformer_random     2612
enformer_low        2295
Name: count, dtype: Int64

In [61]:
# took this from combine_enformer_mpralm.ipynb 

# current dataset: significants niveau: 0.05: 390 => ~0.9% significant
# current dataset: significants niveau: 0.1: 511 => ~1.2% significant
# expected according to Mohan: 5% significant

# number of positive and negative results
matched_pos_effect = matched_significant_results_rare[matched_significant_results_rare['logFC'] > 0]
matched_neg_effect = matched_significant_results_rare[matched_significant_results_rare['logFC'] < 0]

n_tested_vars = tested_bc_MPRAlm_enformer_merged.shape[0]
n_matched_significant = matched_significant_results_rare.shape[0]
n_matched_positive = matched_pos_effect.shape[0]
n_matched_negative = matched_neg_effect.shape[0]
print('---------- With a significants niveau of %s:----------'%(singificants_niveau))
print('Number of significant results after bc_MPRAlm and matching with enformer class file: %s'%(matched_significant_results_rare.shape[0]))
print('Fraction of matchable significant results: %s'%(round(n_matched_significant / n_significant, 3))) 
print('This means %s of the significant results could be matched with enformer class information.\n   Which means if everything works perfect the rest of the signifiant effects is from common variants.'%(round(n_matched_significant / n_significant, 3)))
print('Fraction of significant results: %s'%(round(matched_significant_results_rare.shape[0] / n_tested_vars, 3)))
print('Number of positive effects: %s'%(n_matched_positive))
print('Number of negative effects: %s'%(n_matched_negative))

# significant results enformer class distribution

matched_significant_results_rare[['enformer_class', 'ID']].groupby(['enformer_class' ]).count()

---------- With a significants niveau of 0.05:----------
Number of significant results after bc_MPRAlm and matching with enformer class file: 239
Fraction of matchable significant results: 0.613
This means 0.613 of the significant results could be matched with enformer class information.
   Which means if everything works perfect the rest of the signifiant effects is from common variants.
Fraction of significant results: 0.007
Number of positive effects: 136
Number of negative effects: 103


,ID
enformer_class,
enformer_high,196
enformer_low,15
enformer_random,28


In [62]:
rare_variant_results[['enformer_class', 'ID']].groupby(['enformer_class']).count()

,ID
enformer_class,
enformer_high,13500
enformer_low,2295
enformer_random,2612


In [66]:
rare_variant_results['variant_type'].value_counts()

variant_type
singleton     9306
ultra-rare    9101
Name: count, dtype: int64

In [63]:
tested_bc_MPRAlm_enformer_merged.shape[0]

34526

#### Perform chisquare test

In [65]:
significant_high = 196
significant_not_high = 15+28
all_high = 13500
all_not_high = 2295 + 2612
all_values = all_high + all_not_high


total_sig = n_matched_significant
total_non_sig = rare_variant_results.shape[0] - n_matched_significant
total_high = all_high
total_non_high = all_not_high
table_total = rare_variant_results.shape[0]

# test with chi-square 
from scipy.stats import chisquare
observed = [significant_high, significant_not_high, all_high-significant_high, all_not_high-significant_not_high]
expected = [total_sig * total_high / table_total, total_sig * total_non_high / table_total, total_non_sig * total_high / table_total, total_non_sig * total_non_high / table_total]

chisquare(observed, expected) # Power_divergenceResult(statistic=9.302463802876439, pvalue=0.025528383067005125)
print('interpretation: enformer has a huge number of false positives.')
print('This means the distribution of significant results in the enformer class file is significantly different from the distribution of all results in the enformer class file.')



Power_divergenceResult(statistic=9.302463802876439, pvalue=0.025528383067005125)

#### Bin Analysis

In [74]:
# sort by logFC
logFC_sorted = matched_significant_results_rare[['logFC', 'variant_id', 'enformer_class', 'variant_type', 'gene_type_list', 'DNase_max']].sort_values(by='logFC', ascending=True)
matched_significant_results_rare['abs_logFC'] = matched_significant_results_rare['logFC'].abs()
# sort by logFC (absolut value)
abs_logFC_sorted = matched_significant_results_rare[['logFC', 'variant_id', 'enformer_class', 'variant_type', 'gene_type_list', 'DNase_max', 'abs_logFC']].sort_values(by='abs_logFC', ascending=True)
abs_bins = [abs_logFC_sorted.iloc[i*abs_logFC_sorted.shape[0]//10:(i+1)*abs_logFC_sorted.shape[0]//10] for i in range(10)]

# make equally distant bins of the logFC and plot  number of enformer_high occurences per bin as histogram 
bins = [logFC_sorted.iloc[i*logFC_sorted.shape[0]//10:(i+1)*logFC_sorted.shape[0]//10] for i in range(10)]
# # barplot of numbers of enformer_high occurnces in enformer_class column per bin
# [bins[i].describe for i in range(10)]
# # 495 -0.270634  cardiac_neuro_cava_random:BCL10|ENSG0000014286...   
# #  569  0.268013  cardiac_neuro_cava_random:PHACTR1|ENSG00000112...
# # This means significant logFC is from -0.27 and from 0.27

/tmp/ipykernel_1294/284524763.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched_significant_results_rare['abs_logFC'] = matched_significant_results_rare['logFC'].abs()


In [75]:
# get min max logFC values of bins like: '<bin_min> - <bin_max>' for each bin
bin_description = [str(round(bin['logFC'].min(), 2)) + ' - ' + str(round(bin['logFC'].max(), 2)) for bin in bins]
abs_bin_description = [str(round(bin['abs_logFC'].min(), 2)) + ' - ' + str(round(bin['abs_logFC'].max(), 2)) for bin in abs_bins]
bin_description

['-1.5 - -0.87',
 '-0.84 - -0.65',
 '-0.65 - -0.51',
 '-0.5 - -0.35',
 '-0.35 - 0.41',
 '0.41 - 0.51',
 '0.51 - 0.63',
 '0.64 - 0.76',
 '0.77 - 0.91',
 '0.91 - 1.61']

In [1]:
# plot number_enformer_low and number_enformer_high in same bar plot (for absolut and for range of log2FC values)

import matplotlib.pyplot as plt
import numpy as np
x = np.arange(10)  # the label locations
width = 0.25  # the width of the bars
multiplier = 0

# get the numbers of each group for each bin
number_enformer_high = [bin[bin['enformer_class'] == 'enformer_high'].shape[0] for bin in abs_bins]
number_enformer_low = [bin[bin['enformer_class'] == 'enformer_low'].shape[0] for bin in abs_bins]
number_random = [bin[bin['enformer_class'] == 'random'].shape[0] for bin in abs_bins]
group_numbers = {'enformer_high': number_enformer_high, 'enformer_low': number_enformer_low, 'random': number_random}


# absolut values
fig, ax = plt.subplots(layout='constrained')

for attribute, measurement in group_numbers.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Number of enformer labels')
# ax.set_title('Enformer label distribution for DNase values along the sorted absolute logFC value')
ax.set_title('Enformer label distribution along the sorted absolute log2FC value')
ax.set_xticks(x + width, abs_bin_description, rotation=45)
ax.set_xlabel('absolut Log2FC bin range')
ax.legend(loc='upper left', ncols=3)
ax.set_ylim(0, 40)

plt.show()


# full range of logFC values
number_enformer_high = [bin[bin['enformer_class'] == 'enformer_high'].shape[0] for bin in bins]
number_enformer_low = [bin[bin['enformer_class'] == 'enformer_low'].shape[0] for bin in bins]
number_random = [bin[bin['enformer_class'] == 'random'].shape[0] for bin in bins]
group_numbers = {'enformer_high': number_enformer_high, 'enformer_low': number_enformer_low, 'random': number_random}

fig, ax = plt.subplots(layout='constrained')

for attribute, measurement in group_numbers.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Number of enformer labels')
# ax.set_title('Enformer label distribution for DNase values along the sorted absolute logFC value')
ax.set_title('Enformer label distribution along the sorted log2FC value')
ax.set_xticks(x + width, bin_description, rotation=45)
ax.set_xlabel('Log2FC bin range')
ax.legend(loc='upper left', ncols=3)
ax.set_ylim(0, 40)

plt.show()


NameError: name 'abs_bins' is not defined

### Load the annotations of enformer columns (more precise analysis)

In [8]:
# investigate columns of enformer
import pandas as pd
targets_txt = 'https://raw.githubusercontent.com/calico/basenji/0.5/manuscripts/cross2020/targets_human.txt'
df_targets = pd.read_csv(targets_txt, sep='\t')

In [9]:
df_targets

,index,genome,identifier,file,clip,scale,sum_stat,description
0,0,0,ENCFF833POA,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:cerebellum male adult (27 years) and mal...
1,1,0,ENCFF110QGM,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:frontal cortex male adult (27 years) and...
2,2,0,ENCFF880MKD,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:chorion
3,3,0,ENCFF463ZLQ,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:Ishikawa treated with 0.02% dimethyl sul...
4,4,0,ENCFF890OGQ,/home/drk/tillage/datasets/human/dnase/encode/...,32,2,mean,DNASE:GM03348
...,...,...,...,...,...,...,...,...
5308,5308,0,CNhs14239,/home/drk/tillage/datasets/human/cage/fantom/C...,384,1,sum,CAGE:epithelioid sarcoma cell line:HS-ES-2R
5309,5309,0,CNhs14240,/home/drk/tillage/datasets/human/cage/fantom/C...,384,1,sum,CAGE:squamous cell lung carcinoma cell line:RE...
5310,5310,0,CNhs14241,/home/drk/tillage/datasets/human/cage/fantom/C...,384,1,sum,CAGE:gastric cancer cell line:GSS
5311,5311,0,CNhs14244,/home/drk/tillage/datasets/human/cage/fantom/C...,384,1,sum,CAGE:carcinoid cell line:NCI-H727


In [4]:
# TODO: add here the comparison of the element enformer predictions 